# SolarSense EMS — Industrial Load & LESCO MDI Spike Analysis
### Phase 2 Milestone 5: Exploratory Data Analysis & Baseline Loss Reconciliation
**Location:** Lahore Industrial Area, Punjab, Pakistan  
**Facility:** Textile Spinning & Weaving Mill (500 kW LESCO B3 Sanctioned Ceiling)  
**Period:** July 1, 2025 – October 7, 2026 (464 Days / 11,136 Hourly Observations)

---

## 1. Executive Summary & Problem Formulation
Under the **NEPRA / LESCO B3 Industrial Tariff**, high-voltage consumers face three major cost drivers:
1. **Time-of-Use (ToU) Energy Rates:** 38.00 PKR/kWh off-peak vs. 60.00 PKR/kWh during the 4-hour seasonal peak window.
2. **Maximum Demand Indicator (MDI) Ratchet:** Exceeding the 500 kW sanctioned limit incurs a fine of **1,200.00 PKR per peak kW**, billed on the highest instantaneous peak recorded that calendar month.
3. **Load Shedding / Blackout Diesel Generation:** During feeder outages, unmet factory demand must be served by emergency diesel generators at **95.00 PKR/kWh**.

### The Critical Physics Law: Outage MDI Immunity
A naive baseline calculation previously charged 1,200 PKR/kW penalties during grid blackouts, fabricating over **2.36 Million PKR** in non-existent penalties. This notebook proves that because the LESCO feeder line is de-energized during blackouts (`outage_flag == 1`), the factory meter registers `0.0 kW` grid draw. The factory cannot incur an MDI penalty when the grid is dead.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from src.data_engineering.feature_store import EMSDataRepository, TariffPolicy

repo = EMSDataRepository()
policy = TariffPolicy()
df = repo._df

print("Canonical EMS Dataset Loaded:")
print(f"Total Rows: {df.height:,}")
print(f"Date Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Total Outage Hours: {df['outage_flag'].sum()}")


## 2. Factory Load Distribution & Shift Analytics
The facility operates on a 2-shift production schedule with a lighter maintenance night shift:
- **Shift 1 (Day):** 08:00 – 16:00
- **Shift 2 (Evening / Peak Overlap):** 16:00 – 00:00
- **Night Shift (Reduced Operations):** 00:00 – 08:00


In [ ]:
# Add shift classification
df_shifts = df.with_columns([
    pl.col("timestamp").dt.hour().alias("hour"),
    pl.when(pl.col("timestamp").dt.hour().is_between(8, 15))
    .then(pl.lit("Shift 1 (08-16h)"))
    .when(pl.col("timestamp").dt.hour().is_between(16, 23))
    .then(pl.lit("Shift 2 (16-00h)"))
    .otherwise(pl.lit("Night Shift (00-08h)"))
    .alias("shift")
])

shift_stats = df_shifts.group_by("shift").agg([
    pl.col("factory_load_kw").mean().round(2).alias("mean_load_kw"),
    pl.col("factory_load_kw").median().round(2).alias("median_load_kw"),
    pl.col("factory_load_kw").max().round(2).alias("peak_load_kw"),
    pl.col("factory_load_kw").quantile(0.95).round(2).alias("p95_load_kw"),
]).sort("shift")

print("Factory Load Statistics by Operational Shift:")
print(shift_stats)


## 3. Historical MDI Spike Analysis (> 500 kW Ceiling)
We identify all hours where net grid import breached the sanctioned 500 kW ceiling during **grid-connected operation**.


In [ ]:
mdi_spikes = repo.get_mdi_spikes(threshold_kw=500.0)
print(f"Total Grid-Connected MDI Breaches: {mdi_spikes.height}")
print("Top 10 Worst Historical MDI Violations:")
print(mdi_spikes.head(10))


## 4. Reconciling the 2.36 Million PKR Baseline Discrepancy
Comparing naive accounting (penalizing gross load during blackouts) vs. true physical accounting (enforcing outage MDI immunity and charging diesel at 95 PKR/kWh).


In [ ]:
# Naive gross demand exceedance during blackouts
blackout_exceedances = df.filter(
    (pl.col("outage_flag") == 1) & (pl.col("factory_load_kw") > 500.0)
)

fake_penalties = float(
    ((blackout_exceedances["factory_load_kw"] - 500.0) * float(policy.mdi_penalty_per_kw_pkr)).sum()
)

print(f"Blackout Hours with Gross Load > 500 kW: {blackout_exceedances.height}")
print(f"Phantom MDI Fines Avoided by Physics Invariant: {fake_penalties:,.2f} PKR")
print(f"Actual Grid MDI Violations Counted by LESCO: {mdi_spikes.height}")


## 5. Visualizing Diurnal Load Curve & MDI Breaches
We plot the 24-hour diurnal factory load, solar generation, and the 500 kW sanctioned boundary.


In [ ]:
hourly_profiles = df.group_by(df["timestamp"].dt.hour().alias("hour")).agg([
    pl.col("factory_load_kw").mean().alias("avg_load_kw"),
    pl.col("factory_load_kw").quantile(0.95).alias("p95_load_kw"),
    pl.col("factory_load_kw").max().alias("max_load_kw"),
    pl.col("effective_solar_kw").mean().alias("avg_solar_kw"),
]).sort("hour")

hours = hourly_profiles["hour"].to_numpy()
avg_load = hourly_profiles["avg_load_kw"].to_numpy()
p95_load = hourly_profiles["p95_load_kw"].to_numpy()
max_load = hourly_profiles["max_load_kw"].to_numpy()
avg_solar = hourly_profiles["avg_solar_kw"].to_numpy()

plt.figure(figsize=(12, 6))
plt.plot(hours, max_load, color="firebrick", linestyle="--", label="Historical Max Load (kW)", alpha=0.8)
plt.plot(hours, p95_load, color="darkorange", label="95th Percentile Load (kW)", linewidth=2)
plt.plot(hours, avg_load, color="navy", label="Average Factory Load (kW)", linewidth=2.5)
plt.plot(hours, avg_solar, color="gold", label="Average Rooftop Solar PV (kW)", linewidth=2)

# Sanctioned ceiling
plt.axhline(500.0, color="crimson", linewidth=2.5, linestyle="-", label="LESCO 500 kW Sanctioned Limit")
plt.fill_between(hours, 500.0, np.maximum(500.0, max_load), color="crimson", alpha=0.2, label="MDI Penalty Region (>500 kW)")

plt.title("SolarSense EMS — 24-Hour Diurnal Factory Demand & 500 kW LESCO Ceiling", fontsize=14, fontweight="bold")
plt.xlabel("Hour of Day (Asia/Karachi PKT)", fontsize=12)
plt.ylabel("Power (kW)", fontsize=12)
plt.xticks(range(0, 24))
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("../docs/samples/load_and_mdi_spikes.png", dpi=150)
print("Plot saved to docs/samples/load_and_mdi_spikes.png")
plt.close()


## 6. Conclusions & Implications for MILP Battery Optimizer
1. **Shift 2 Vulnerability:** Peak factory demand coincides directly with the LESCO seasonal peak window (17:00 – 21:00 in winter, 18:00 – 22:00 in spring/fall), creating double financial exposure (60 PKR/kWh + 1,200 PKR/kW MDI risk).
2. **P90 Safety Margin:** The battery optimizer must dispatch battery discharge based on the $p90$ load forecast during Shift 2 to prevent any draw $>500	ext{ kW}$.
3. **Outage Islanding:** When blackout risk is elevated, the battery must remain at $\ge 90\%$ state-of-charge to eliminate the 95 PKR/kWh diesel penalty.
